# Agentic RAG
Let's build a simple agent that can search the web. This agent will retrieve information and synthesize responses to answer queries. 
Kitambo it was just regurgitating but now it gives tailored responses

In [ ]:
pip install --upgrade litellm mcp

In [ ]:
from smolagents import ToolCallingAgent,DuckDuckGoSearchTool, LiteLLMModel
from getpass import getpass
#initialize search tool
search_tool= DuckDuckGoSearchTool()

# Set your Groq key
import os
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

#initialize model
model = LiteLLMModel(
    model_id="groq/openai/gpt-oss-120b", # Or another supported model
    temperature=0.1
)



agent=ToolCallingAgent(
    tools=[search_tool],
    model=model
)

agent.run("Search for luxury Barbie themed party ideas, including decorations, entertainment, and catering."
)


The agent follows this process:

1. Analyzes the Request: Alfred’s agent identifies the key elements of the query—luxury superhero-themed party planning, with focus on decor, entertainment, and catering.
2. Performs Retrieval: The agent leverages WebSearchTool to search for the most relevant and up-to-date information, ensuring it aligns with Alfred’s refined preferences for a luxurious event.
3. Synthesizes Information: After gathering the results, the agent processes them into a cohesive, actionable plan for Alfred, covering all aspects of the party.
4. Stores for Future Reference: The agent stores the retrieved information for easy access when planning future events, optimizing efficiency in subsequent tasks.

## Custom Knowledge Base Tool
* For specialized tasks, a custom knowledge base can be invaluable. 
* Let's create a tool that queries a vector database of technical documentation or specialized knowledge. Using semantic search, the agent can find the most relevant information for Alfred's needs.
* **Vector database** is a specialized storage system that saves data as vectors (lists of numbers) to enable searching by meaning and similarity rather than exact keyword matches.

In [ ]:
!pip install langchain-community rank_bm25

In [ ]:
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from smolagents import Tool
from langchain_community.retrievers import BM25Retriever
from smolagents import CodeAgent, InferenceClientModel

class PartyPlanningRetrieverTool(Tool):
    name = "party_planning_retriever"
    description = "Uses semantic search to retrieve relevant party planning ideas for Alfred’s superhero-themed party at Wayne Manor."
    inputs = {
        "query": {
            "type": "string",
            "description": "The query to perform. This should be a query related to party planning or superhero themes.",
        }
    }
    output_type = "string"

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)
        self.retriever = BM25Retriever.from_documents(
            docs, k=5  # Retrieve the top 5 documents
        )

    def forward(self, query: str) -> str:
        assert isinstance(query, str), "Your search query must be a string"

        docs = self.retriever.invoke(
            query,
        )
        return "\nRetrieved ideas:\n" + "".join(
            [
                f"\n\n===== Idea {str(i)} =====\n" + doc.page_content
                for i, doc in enumerate(docs)
            ]
        )

# Simulate a knowledge base about party planning
party_ideas = [
    {"text": "A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.", "source": "Party Ideas 1"},
    {"text": "Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.", "source": "Entertainment Ideas"},
    {"text": "For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'", "source": "Catering Ideas"},
    {"text": "Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.", "source": "Decoration Ideas"},
    {"text": "Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.", "source": "Entertainment Ideas"}
]

source_docs = [
    Document(page_content=doc["text"], metadata={"source": doc["source"]})
    for doc in party_ideas
]

# Split the documents into smaller chunks for more efficient search
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
    strip_whitespace=True,
    separators=["\n\n", "\n", ".", " ", ""],
)
docs_processed = text_splitter.split_documents(source_docs)

# Create the retriever tool
party_planning_retriever = PartyPlanningRetrieverTool(docs_processed)

# Initialize the agent
agent = CodeAgent(tools=[party_planning_retriever], model=InferenceClientModel())

# Example usage
response = agent.run(
    "Find ideas for a luxury superhero-themed party, including entertainment, catering, and decoration options."
)

print(response)

In [ ]:
This enhanced agent can:

First check the documentation for relevant information
Combine insights from the knowledge base
Maintain conversation context through memory

In [ ]:
Enhanced Retrieval Capabilities
When building agentic RAG systems, the agent can employ sophisticated strategies like:

Query Reformulation: Instead of using the raw user query, the agent can craft optimized search terms that better match the target documents
Query Decomposition: Instead of using the user query directly, if it contains multiple pieces of information to query, it can be decomposed to multiple queries
Query Expansion: Somehow similar to Query Reformulation but done multiple times to put the query in multiple wordings to query them all
Reranking: Using Cross-Encoders to assign more comprehensive and semantic relevance scores between retrieved documents and search query
Multi-Step Retrieval: The agent can perform multiple searches, using initial results to inform subsequent queries
Source Integration: Information can be combined from multiple sources like web search and local documentation
Result Validation: Retrieved content can be analyzed for relevance and accuracy before being included in responses